# Feature Store with MinMaxScaler

This notebook demonstrates how to create a Snowflake Feature Store with MinMaxScaler transformation from sklearn.

In [39]:
!pip install snowflake-ml-python scikit-learn

import snowflake.snowpark as snowpark
from snowflake.snowpark import Session
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode
from snowflake.snowpark.functions import col
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [41]:
session = snowpark.context.get_active_session()

In [42]:
USE DATABASE AICOLLEGE;
USE SCHEMA PUBLIC;
USE ROLE AICOLLEGE;
USE WAREHOUSE AICOLLEGE;
SELECT CURRENT_ROLE();
SELECT CURRENT_WAREHOUSE();


In [43]:
sample_data = pd.DataFrame({
    'CUSTOMER_ID': [1, 2, 3, 4, 5],
    'AGE': [25, 45, 35, 50, 28],
    'ANNUAL_INCOME': [50000, 80000, 65000, 95000, 55000],
    'CREDIT_SCORE': [650, 720, 680, 750, 670],
    'PURCHASE_AMOUNT': [1200, 2500, 1800, 3200, 1400]
})

source_df = session.create_dataframe(sample_data)
source_df.write.mode('overwrite').save_as_table('RAW_CUSTOMER_DATA')

print("Sample data created:")
source_df.show()

In [44]:
fs = FeatureStore(
    session=session,
    database=session.get_current_database(),
    name="CUSTOMER_FEATURE_STORE",
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

print("Feature store created successfully!")

In [45]:
customer_entity = Entity(
    name="customer",
    join_keys=['CUSTOMER_ID'],
    desc="Customer entity"
)

fs.register_entity(customer_entity)
print("Entity registered!")

In [49]:
source_table = session.table('RAW_CUSTOMER_DATA')

features_to_scale = ['AGE', 'ANNUAL_INCOME', 'CREDIT_SCORE', 'PURCHASE_AMOUNT']

pandas_df = source_table.to_pandas()

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(pandas_df[features_to_scale])

scaled_df = pandas_df.copy()
for i, col_name in enumerate(features_to_scale):
    scaled_df[f'{col_name}_SCALED'] = scaled_features[:, i]

print(scaled_df.dtypes)

feature_df = session.create_dataframe(scaled_df[
    ['CUSTOMER_ID', 'AGE_SCALED', 'ANNUAL_INCOME_SCALED', 'CREDIT_SCORE_SCALED', 'PURCHASE_AMOUNT_SCALED']
])

feature_df.write.mode('overwrite').save_as_table('AICOLLEGE.CUSTOMER_FEATURE_STORE.CUSTOMER_SCALED_FEATURES_TABLE')

feature_df = session.table('CUSTOMER_SCALED_FEATURES_TABLE')

print("Transformed features with MinMaxScaler:")
feature_df.show()

In [50]:
feature_df.dtypes

In [51]:
customer_fv = FeatureView(
    name="customer_scaled_features",
    entities=[customer_entity],
    feature_df=feature_df,
    refresh_freq='5 minutes',
    desc="Customer features scaled using MinMaxScaler"
)

customer_fv = customer_fv.attach_feature_desc({
    'age_scaled': 'Age normalized to [0,1] range using MinMaxScaler',
    'annual_income_scaled': 'Annual income normalized to [0,1] range',
    'credit_score_scaled': 'Credit score normalized to [0,1] range',
    'purchase_amount_scaled': 'Purchase amount normalized to [0,1] range'
})

print("Feature view created!")

In [52]:
registered_fv = fs.register_feature_view(
    feature_view=customer_fv,
    version='v1',
    block=True
)

print("Feature view registered in feature store!")

In [ ]:
retrieved_fv = fs.get_feature_view(
    name='customer_scaled_features',
    version='v1'
)

print("Retrieved feature view from feature store:")
print(retrieved_fv)

In [ ]:
print("All feature views in the feature store:")
fs.list_feature_views().show()

In [ ]:
spine_df = session.create_dataframe([
    (1,), (2,), (3,), (4,), (5,)
], schema=['customer_id'])

training_data = fs.generate_dataset(
    spine_df=spine_df,
    features=[retrieved_fv],
    name='customer_training_data'
)

print("Training dataset with scaled features:")
training_data.read.to_snowpark_dataframe().show()

## Using receipt from [Build an End-to-End ML Workflow in Snowflake](https://www.snowflake.com/en/developers/guides/end-to-end-ml-workflow/?index=..%2F..index#1)

In [ ]:
--select * from E2E_SNOW_MLOPS_DB.MLOPS_SCHEMA.MORTGAGE_LENDING_DEMO_DATA LIMIT 10;
CREATE TABLE AICOLLEGE.PUBLIC.MORTGAGE_LENDING_DEMO_DATA AS
SELECT * FROM E2E_SNOW_MLOPS_DB.MLOPS_SCHEMA.MORTGAGE_LENDING_DEMO_DATA;

In [ ]:
from datetime import datetime

# Snowpark session
from snowflake.snowpark import DataFrame
from snowflake.snowpark.functions import col, to_timestamp, min, max, month, dayofweek, dayofyear, avg, date_add, sql_expr
from snowflake.snowpark.types import IntegerType
from snowflake.snowpark import Window


In [ ]:
try:
    print("Reading table data...")
    df = session.table("MORTGAGE_LENDING_DEMO_DATA")
    df.show(5)
except:
    print("Table not found! Uploading data to snowflake table")


In [ ]:
df.select(min('TS'), max('TS'))

In [ ]:
#Create a dict with keys for feature names and values containing transform code

feature_eng_dict = dict()

#Timstamp features
feature_eng_dict["TIMESTAMP"] = date_add(to_timestamp("TS"), 1) #, timedelta.days-1)
feature_eng_dict["MONTH"] = month("TIMESTAMP")
feature_eng_dict["DAY_OF_YEAR"] = dayofyear("TIMESTAMP") 
feature_eng_dict["DOTW"] = dayofweek("TIMESTAMP")

# df= df.with_columns(feature_eng_dict.keys(), feature_eng_dict.values())

#Income and loan features
feature_eng_dict["LOAN_AMOUNT"] = col("LOAN_AMOUNT_000s")*1000
feature_eng_dict["INCOME"] = col("APPLICANT_INCOME_000s")*1000
feature_eng_dict["INCOME_LOAN_RATIO"] = col("INCOME")/col("LOAN_AMOUNT")

county_window_spec = Window.partition_by("COUNTY_NAME")
feature_eng_dict["MEAN_COUNTY_INCOME"] = avg("INCOME").over(county_window_spec)
feature_eng_dict["HIGH_INCOME_FLAG"] = (col("INCOME")>col("MEAN_COUNTY_INCOME")).astype(IntegerType())

feature_eng_dict["AVG_THIRTY_DAY_LOAN_AMOUNT"] =  sql_expr("""AVG(LOAN_AMOUNT) OVER (PARTITION BY COUNTY_NAME ORDER BY TIMESTAMP  
                                                            RANGE BETWEEN INTERVAL '30 DAYS' PRECEDING AND CURRENT ROW)""")

df = df.with_columns(feature_eng_dict.keys(), feature_eng_dict.values())
df.show(3)

In [ ]:
df.columns
#df.dtypes
#df.explain()

In [ ]:
#source_table = session.table('RAW_CUSTOMER_DATA')

features_to_scale = ['LOAN_AMOUNT', 'AVG_THIRTY_DAY_LOAN_AMOUNT']

pandas_df = df['LOAN_ID', 'LOAN_AMOUNT', 'AVG_THIRTY_DAY_LOAN_AMOUNT']

#pandas_df.dtypes

pandas_df1 = pandas_df.to_pandas()

pandas_df1.dtypes

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(pandas_df1[features_to_scale])

scaled_df = pandas_df1.copy()
for i, col_name in enumerate(features_to_scale):
    scaled_df[f'{col_name}_SCALED'] = scaled_features[:, i]

print(scaled_df.dtypes)


feature_df = session.create_dataframe(scaled_df[
    ['LOAN_ID', 'LOAN_AMOUNT_SCALED', 'AVG_THIRTY_DAY_LOAN_AMOUNT_SCALED']
])

feature_df.write.mode('overwrite').save_as_table('AICOLLEGE.CUSTOMER_FEATURE_STORE.MORTGAGE_LENDING_DEMO_DATA')

feature_df_show = session.table('AICOLLEGE.CUSTOMER_FEATURE_STORE.MORTGAGE_LENDING_DEMO_DATA')

print("Transformed features with MinMaxScaler:")
feature_df_show.show()

In [ ]:
fs_loan = FeatureStore(
    session=session,
    database=session.get_current_database(),
    name="CUSTOMER_FEATURE_STORE",
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

print("Feature store created successfully!")



In [ ]:
fs_loan.list_entities()

In [ ]:
loan_id_entity = Entity(
        name = "LOAN_ENTITY",
        join_keys = ["LOAN_ID"],
        desc = "Features defined on a per loan level")
#register
fs_loan.register_entity(loan_id_entity)
print("Registered new entity")

In [ ]:
#Create a dataframe with just the ID, timestamp, and engineered features. We will use this to define our feature view
feature_df = df.select(["LOAN_ID"]+list(feature_eng_dict.keys()))
feature_df.show(5)

In [ ]:
customer_fv = FeatureView(
    name="customer_scaled_features",
    entities=[customer_entity],
    feature_df=feature_df,
    refresh_freq='5 minutes',
    desc="Customer features scaled using MinMaxScaler"
)

customer_fv = customer_fv.attach_feature_desc({
    'age_scaled': 'Age normalized to [0,1] range using MinMaxScaler',
    'annual_income_scaled': 'Annual income normalized to [0,1] range',
    'credit_score_scaled': 'Credit score normalized to [0,1] range',
    'purchase_amount_scaled': 'Purchase amount normalized to [0,1] range'
})

print("Feature view created!")